# Clean13 retrain — notebook v2 với mẫu âm thực tế

Thêm hai input Kaggle: smoke_detection_iot_transient_v1.csv và field_measurements.csv đi kèm notebook này. Run All để kiểm tra ba cấu hình, quét ngưỡng, chấm phiên giữ riêng và xuất model.

Giữ đúng 13 đặc trưng động. Lỗi cần sửa là model cũ coi std PM1.0 khoảng 0,548 do dao động nguyên đơn vị là khởi phát. Notebook bổ sung mẫu âm thật và thử thêm một bản sao dữ liệu nguồn có PM được làm tròn trước khi tính đặc trưng. Lượng tử hóa chỉ dùng để tạo dữ liệu tăng cường khi train, công thức suy luận hiện tại vẫn giữ nguyên.

Train âm thực tế từ 17/11. Phiên 12/11 chỉ dùng validation/chọn ngưỡng. Phiên 20/11 chỉ được chấm sau khi fit và chọn ngưỡng. Phiên 20/11 đã được xem trong bước chẩn đoán nên kết quả vẫn là kiểm tra khắc phục, cần thêm phiên tương lai chưa quan sát để đánh giá cuối.

Các cửa sổ khởi phát trong CSV nguồn là nhãn suy ra từ cảm biến. Phiên khói giấy được chấm riêng như challenge chưa gán nhãn cháy; không tự động gán cả phiên thành dương.

Mặc định xuất field_clean13 sau phép đối chiếu validation vì bổ sung mẫu âm thật đã xử lý lỗi nhảy điểm và giữ F1 nguồn tốt hơn bản tăng cường PM trong lượt thử này. Có thể đổi FINAL_VARIANT để xuất bản tăng cường.


In [1]:
from pathlib import Path
import hashlib
import json
import platform
import zipfile

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss, confusion_matrix,
    f1_score, log_loss, precision_score, recall_score, roc_auc_score,
)

try:
    from IPython.display import FileLink, display
except ImportError:
    FileLink = None
    display = print

SOURCE_PATH = None
FIELD_PATH = None
OUTPUT_DIR = Path('/kaggle/working/clean13_field_retrain_v2') if Path('/kaggle/working').exists() else Path('output/analysis/clean13_field_retrain_v2')
FINAL_VARIANT = 'field_clean13'
TARGET_RECALL = .80
THRESHOLDS = np.round(np.arange(.05, .251, .01), 2)
REFERENCE_THRESHOLD = .25
PARAMETERS = dict(n_estimators=150, max_depth=6, num_leaves=31, learning_rate=.05, random_state=42, n_jobs=2, verbosity=-1)
FEATURES = [
    'd_temp_1', 'd_temp_5', 'd_hum_1', 'd_hum_5',
    'd_pm1_1', 'd_pm1_5', 'd_pm25_1', 'd_pm25_5',
    'temp_roll_5_std', 'hum_roll_5_std', 'pm1_roll_5_std',
    'pm25_roll_5_std', 'pm_ratio',
]
SENSORS = {'temperature_c': 'temp', 'humidity_pct': 'hum', 'pm1_ug_m3': 'pm1', 'pm25_ug_m3': 'pm25'}
NORMAL_END_CNT = {1: 3900, 5: 3900, 2: 3172, 3: 3172, 4: 15}
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

def locate(filename, configured, local_relative):
    if configured is not None:
        return Path(configured)
    paths = list(Path('/kaggle/input').rglob(filename)) if Path('/kaggle/input').exists() else []
    for root in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        paths.extend([root / filename, root / local_relative / filename])
    paths = sorted({p.resolve() for p in paths if p.is_file()})
    if len(paths) != 1:
        raise ValueError(f'Set the path for {filename}. Found {paths}')
    return paths[0]

def boolean(series):
    parsed = series.astype(str).str.lower().map({'true': True, 'false': False, '1': True, '0': False})
    if parsed.isna().any():
        raise ValueError(f'Invalid boolean values in {series.name}')
    return parsed.astype(bool)

source_path = locate('smoke_detection_iot_transient_v1.csv', SOURCE_PATH, 'Model/smoke_transient_v1')
field_path = locate('field_measurements.csv', FIELD_PATH, 'Model/clean13_field_retrain_v2')
source = pd.read_csv(source_path, float_precision='round_trip').sort_values(['session_id', 'UTC', 'CNT']).reset_index(drop=True)
field = pd.read_csv(field_path, float_precision='round_trip')
source['training_eligible'] = boolean(source.training_eligible)
field['use_sample'] = boolean(field.use_sample)
if not source.Fire_Alarm_Transient.isin([0, 1]).all():
    raise ValueError('Invalid source labels')
if not source.session_id.map({1: 'A', 5: 'A', 2: 'B', 3: 'B', 4: 'C'}).eq(source.source_group).all():
    raise ValueError('Invalid paired-session grouping')
if field.groupby('source_session').role.nunique().max() != 1:
    raise ValueError('One physical session was assigned multiple roles')
role_sessions = {role: set(part.source_session) for role, part in field.groupby('role')}
assert role_sessions['train_negative'].isdisjoint(role_sessions['validation_negative'])
assert role_sessions['train_negative'].isdisjoint(role_sessions['heldout_negative'])
assert role_sessions['validation_negative'].isdisjoint(role_sessions['heldout_negative'])
print('Field session roles before history filtering:')
display(field.groupby(['source_session', 'role']).use_sample.agg(rows='size', selected='sum'))


Field session roles before history filtering:
                                          rows  selected
source_session       role                               
cool_20261117        train_negative        646       643
paper_smoke_20261117 smoke_challenge       361       298
room_20261112        validation_negative   635       635
room_20261117        train_negative       1109       900
room_20261120        heldout_negative      101       101
warm_20261117        train_negative        326       275


## Đặc trưng nhân quả và dữ liệu tăng cường

Mỗi vector dùng tối đa sáu mẫu liên tục; std dùng năm mẫu và ddof=1. Tính trên chuỗi đầy đủ rồi mới lọc hàng dùng học. Ở source, giữ quy tắc cleaning đã có và warmup 10 mẫu để đối chiếu cùng bản trước. Log thật cần tối thiểu sáu mẫu lịch sử, reset nếu đổi phiên/boot, mất seq hoặc uptime gián đoạn trên 2,5 giây.

field_augmented13 học cả bản source gốc và bản PM làm tròn về số nguyên; mỗi bản source có trọng số 0,5 để tổng trọng số source giữ nguyên. Mẫu âm thật có trọng số 1. Các bản tăng cường vẫn nằm trong cùng nhóm gốc, không tách chúng sang validation của cùng fold.


In [2]:
def make_features(frame, blocks, quantize_pm=False):
    values = frame[list(SENSORS)].copy()
    if quantize_pm:
        values[['pm1_ug_m3', 'pm25_ug_m3']] = values[['pm1_ug_m3', 'pm25_ug_m3']].round(0)
    if not np.isfinite(values.to_numpy(dtype=float)).all():
        raise ValueError('Nonfinite sensor input')
    result = pd.DataFrame(index=frame.index)
    for sensor, prefix in SENSORS.items():
        grouped = values[sensor].groupby(blocks, sort=False)
        result[f'd_{prefix}_1'] = grouped.diff(1).fillna(0.)
        result[f'd_{prefix}_5'] = grouped.diff(5).fillna(0.)
        result[f'{prefix}_roll_5_std'] = grouped.transform(lambda s: s.rolling(5, min_periods=1).std(ddof=1).fillna(0.))
    result['pm_ratio'] = (values.pm25_ug_m3 + .001) / (values.pm1_ug_m3 + .001)
    return result[FEATURES]

source = source.rename(columns={'Temperature[C]': 'temperature_c', 'Humidity[%]': 'humidity_pct', 'PM1.0': 'pm1_ug_m3', 'PM2.5': 'pm25_ug_m3'})
source_reset = source.session_id.ne(source.session_id.shift()) | source.UTC.diff().ne(1) | source.CNT.diff().ne(1)
source_block = source_reset.cumsum()
Xs = make_features(source, source_block)
Xq = make_features(source, source_block, quantize_pm=True)
ys = source.Fire_Alarm_Transient.astype(int)
source_groups = source.source_group
source_mask = (source.training_eligible & source.groupby(source_block).cumcount().ge(10)
               & (ys.eq(1) | source.CNT.le(source.session_id.map(NORMAL_END_CNT))))
assert source_mask.sum() == 14864 and ys.loc[source_mask].sum() == 832

field_reset = (field.source_session.ne(field.source_session.shift()) | field.boot_id.ne(field.boot_id.shift())
               | field.seq.diff().ne(1) | field.uptime_ms.diff().gt(2500)
               | field.sensor_status.ne('OK') | field.sensor_status.shift().ne('OK'))
field_block = field_reset.cumsum()
Xf = make_features(field, field_block)
field_ready = field.use_sample & field.groupby(field_block).cumcount().ge(5) & field.sensor_status.eq('OK')
train_field = field_ready & field.role.eq('train_negative')
validation_field = field_ready & field.role.eq('validation_negative')
heldout_field = field_ready & field.role.eq('heldout_negative')
challenge_field = field_ready & field.role.eq('smoke_challenge')
assert field.loc[train_field | validation_field | heldout_field, 'training_label'].eq(0).all()
assert field.loc[challenge_field, 'training_label'].isna().all()
assert len(FEATURES) == 13
print('Scorable field samples:')
display(field.loc[field_ready].groupby(['source_session', 'role']).size().to_frame('rows'))
print('Source counts:')
display(source.loc[source_mask].groupby('source_group').Fire_Alarm_Transient.agg(rows='size', positive='sum'))


Scorable field samples:
                                          rows
source_session       role                     
cool_20261117        train_negative        636
paper_smoke_20261117 smoke_challenge       298
room_20261112        validation_negative   630
room_20261117        train_negative        900
room_20261120        heldout_negative       96
warm_20261117        train_negative        275
Source counts:
              rows  positive
source_group                
A             8440       658
B             6352       108
C               72        66


In [3]:
def scores_metrics(truth, score):
    return {'rows': len(truth), 'positive': int(np.sum(truth)),
            'roc_auc': float(roc_auc_score(truth, score)),
            'average_precision': float(average_precision_score(truth, score)),
            'log_loss': float(log_loss(truth, score, labels=[0, 1])),
            'brier_score': float(brier_score_loss(truth, score))}

def prediction_metrics(truth, predicted):
    tn, fp, fn, tp = map(int, confusion_matrix(truth, predicted, labels=[0, 1]).ravel())
    return {'accuracy': float(accuracy_score(truth, predicted)),
            'precision': float(precision_score(truth, predicted, zero_division=0)),
            'recall': float(recall_score(truth, predicted, zero_division=0)),
            'f1': float(f1_score(truth, predicted, zero_division=0)),
            'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp}

def sweep_thresholds(truth, score):
    return pd.DataFrame([{'threshold': float(t), **prediction_metrics(truth, np.asarray(score) >= t)} for t in THRESHOLDS])

def choose(table):
    feasible = table.loc[table.recall.ge(TARGET_RECALL)]
    candidates = feasible if len(feasible) else table
    best = candidates.sort_values(['f1', 'precision', 'threshold'], ascending=False).iloc[0]
    return {'threshold': float(best.threshold), 'recall_target_met': bool(len(feasible)),
            'max_recall_in_range': float(table.recall.max()), 'precision': float(best.precision),
            'recall': float(best.recall), 'f1': float(best.f1)}

def fit_variant(variant, included_groups):
    mask = source_mask & source_groups.isin(included_groups)
    if variant == 'field_augmented13':
        parts = [Xs.loc[mask], Xq.loc[mask]]
        labels = [ys.loc[mask], ys.loc[mask]]
        weights = [np.full(mask.sum(), .5), np.full(mask.sum(), .5)]
    else:
        parts, labels, weights = [Xs.loc[mask]], [ys.loc[mask]], [np.ones(mask.sum())]
    if variant != 'source_clean13':
        parts.append(Xf.loc[train_field])
        labels.append(pd.Series(0, index=field.index[train_field]))
        weights.append(np.ones(train_field.sum()))
    matrix = pd.concat(parts, ignore_index=True)
    target = pd.concat(labels, ignore_index=True)
    assert target.nunique() == 2
    estimator = lgb.LGBMClassifier(**PARAMETERS)
    estimator.fit(matrix, target, sample_weight=np.concatenate(weights))
    return estimator

def predict(estimator, matrix):
    return estimator.predict_proba(matrix)[:, list(estimator.classes_).index(1)]

def matrix_table(result):
    return pd.DataFrame([[result['TN'], result['FP']], [result['FN'], result['TP']]],
                         index=['Thực tế 0', 'Thực tế 1'], columns=['Dự đoán 0', 'Dự đoán 1'])


## Validation theo nhóm và chọn ngưỡng

source_clean13 là đối chứng cũ. field_clean13 thêm mẫu âm thật. field_augmented13 thêm cả mẫu âm thật và bản sao source được lượng tử hóa PM. Các siêu tham số model giữ cố định.

Mỗi nhóm A/B/C được chấm bởi model chưa học nhóm đó. Ngưỡng phát triển được chọn từ các điểm OOF source và phiên âm validation 12/11. Điểm phiên 12/11 trong bước này là trung bình từ ba model validation. Phiên giữ riêng 20/11 không tham gia fit hoặc chọn ngưỡng.

Hai cấu hình có dữ liệu thật còn chạy chọn ngưỡng lồng nhau. Với mỗi nhóm ngoài, tạo OOF chỉ trong hai nhóm train, cộng phiên âm validation, chọn ngưỡng rồi chấm nhóm ngoài. Các chỉ số này hỗ trợ đọc mức chênh so với kết quả phát triển.


In [4]:
variants = ['source_clean13', 'field_clean13', 'field_augmented13']
experiments = {}
source_table_rows = []
for variant in variants:
    oof = pd.Series(np.nan, index=source.index[source_mask])
    oof_quantized = pd.Series(np.nan, index=oof.index)
    validation_predictions = []
    nested_prediction = pd.Series(False, index=oof.index)
    nested_rows = []
    for held in ['A', 'B', 'C']:
        available = [g for g in ['A', 'B', 'C'] if g != held]
        outer_val = source_mask & source_groups.eq(held)
        estimator = fit_variant(variant, available)
        p = predict(estimator, Xs.loc[outer_val])
        pq = predict(estimator, Xq.loc[outer_val])
        oof.loc[source.index[outer_val]] = p
        oof_quantized.loc[source.index[outer_val]] = pq
        validation_predictions.append(predict(estimator, Xf.loc[validation_field]))
        source_table_rows.append({'variant': variant, 'held_out_group': held,
                                  **scores_metrics(ys.loc[outer_val], p),
                                  **prediction_metrics(ys.loc[outer_val], p >= REFERENCE_THRESHOLD)})
        if variant != 'source_clean13':
            inner_mask = source_mask & source_groups.isin(available)
            inner_oof = pd.Series(np.nan, index=source.index[inner_mask])
            inner_field_scores = []
            for inner_held in available:
                inner_model = fit_variant(variant, [g for g in available if g != inner_held])
                inner_val = source_mask & source_groups.eq(inner_held)
                inner_oof.loc[source.index[inner_val]] = predict(inner_model, Xs.loc[inner_val])
                inner_field_scores.append(predict(inner_model, Xf.loc[validation_field]))
            assert inner_oof.notna().all()
            cal_y = np.concatenate([ys.loc[inner_oof.index], np.zeros(validation_field.sum(), dtype=int)])
            cal_score = np.concatenate([inner_oof.to_numpy(), np.mean(inner_field_scores, axis=0)])
            selected = choose(sweep_thresholds(cal_y, cal_score))
            nested = p >= selected['threshold']
            nested_prediction.loc[source.index[outer_val]] = nested
            nested_rows.append({'held_out_group': held, 'threshold': selected['threshold'],
                                'calibration_target_met': selected['recall_target_met'],
                                **prediction_metrics(ys.loc[outer_val], nested)})
        print(f'Completed {variant}, group {held}', flush=True)
    assert oof.notna().all() and oof_quantized.notna().all()
    validation_mean = np.mean(validation_predictions, axis=0)
    development_y = np.concatenate([ys.loc[oof.index], np.zeros(validation_field.sum(), dtype=int)])
    development_score = np.concatenate([oof.to_numpy(), validation_mean])
    sweep = sweep_thresholds(development_y, development_score)
    selection = choose(sweep)
    experiments[variant] = {
        'oof': oof, 'oof_quantized': oof_quantized, 'sweep': sweep, 'selection': selection,
        'source_scores': scores_metrics(ys.loc[oof.index], oof),
        'source_classification': prediction_metrics(ys.loc[oof.index], oof.ge(selection['threshold'])),
        'quantized_source_classification': prediction_metrics(ys.loc[oof_quantized.index], oof_quantized.ge(selection['threshold'])),
        'nested_source_classification': prediction_metrics(ys.loc[oof.index], nested_prediction) if variant != 'source_clean13' else None,
        'nested_rows': nested_rows,
        'validation_negative_false_positive_rate': float(np.mean(validation_mean >= selection['threshold'])),
    }

comparison = pd.DataFrame([{'variant': name, **e['source_scores'], 'threshold': e['selection']['threshold'],
                            **e['source_classification'], 'validation_field_fpr': e['validation_negative_false_positive_rate']}
                           for name, e in experiments.items()])
display(comparison.round(6))
display(pd.DataFrame(source_table_rows).round(6))
for name in ['field_clean13', 'field_augmented13']:
    print(name, 'nested source classification')
    display(pd.DataFrame(experiments[name]['nested_rows']).round(6))
    display(pd.DataFrame([experiments[name]['nested_source_classification']]).round(6))
print('Source OOF after PM quantization, each paired group still outside its training fold:')
display(pd.DataFrame([{'variant': name, **e['quantized_source_classification']} for name, e in experiments.items()]).round(6))


Completed source_clean13, group A
Completed source_clean13, group B
Completed source_clean13, group C
Completed field_clean13, group A
Completed field_clean13, group B
Completed field_clean13, group C
Completed field_augmented13, group A
Completed field_augmented13, group B
Completed field_augmented13, group C
             variant   rows  positive   roc_auc  average_precision  log_loss  brier_score  threshold  accuracy  precision    recall        f1     TN   FP  FN   TP  validation_field_fpr
0     source_clean13  14864       832  0.949329           0.905897  0.085395     0.016224       0.25  0.972955   0.698706  0.908654  0.789969  13706  326  76  756              0.673016
1      field_clean13  14864       832  0.942107           0.882150  0.094306     0.017225       0.25  0.973560   0.706103  0.903846  0.792831  13719  313  80  752              0.000000
2  field_augmented13  14864       832  0.942775           0.831045  0.096564     0.018219       0.25  0.973628   0.711946  0.888221  

## Chấm log thực tế sau khi fit và chọn ngưỡng

Phiên 20/11 chỉ được dùng ở cell này để chấm, chưa dùng ở các bước fit/chọn ngưỡng phía trên. Các phiên train được ghi rõ vai trò trong bảng. FPR trên phiên train là khả năng khớp train; FPR phiên giữ riêng mới là kiểm tra chuyển sang phiên khác.

count_ge_099 đếm các điểm quá tự tin đã thấy trong lỗi cũ. vote_3_of_5_positive là phép kiểm tra tầng quyết định tham khảo, được tính riêng từ điểm, không tham gia fit hoặc chọn ngưỡng.


In [5]:
def field_summary(part, scores, threshold):
    reset = part.source_session.ne(part.source_session.shift()) | part.boot_id.ne(part.boot_id.shift()) | part.seq.diff().ne(1)
    flags = pd.Series(np.asarray(scores) >= threshold, index=part.index)
    votes = flags.groupby(reset.cumsum()).transform(lambda s: s.rolling(5, min_periods=5).sum()).ge(3)
    known_negative = part.role.ne('smoke_challenge').all()
    return {'rows': len(part), 'threshold': threshold, 'score_max': float(np.max(scores)),
            'score_p99': float(np.quantile(scores, .99)), 'count_ge_099': int(np.sum(np.asarray(scores) >= .99)),
            'direct_positive': int(flags.sum()), 'vote_3_of_5_positive': int(votes.sum()),
            'false_positive_rate': float(flags.mean()) if known_negative else None}

final_models = {}
field_result_rows = []
field_score_rows = []
for name in variants:
    estimator = fit_variant(name, ['A', 'B', 'C'])
    final_models[name] = estimator
    threshold = experiments[name]['selection']['threshold']
    for session, part in field.loc[field_ready].groupby('source_session', sort=False):
        p = predict(estimator, Xf.loc[part.index])
        field_result_rows.append({'variant': name, 'session': session, 'role': part.role.iloc[0], **field_summary(part, p, threshold)})
        score_part = part[['source_session', 'role', 'boot_id', 'seq', 'training_label']].copy()
        score_part['variant'] = name
        score_part['probability_onset'] = p
        field_score_rows.append(score_part)
field_results = pd.DataFrame(field_result_rows)
display(field_results.round(6))
selected = experiments[FINAL_VARIANT]
final_model = final_models[FINAL_VARIANT]
threshold = selected['selection']['threshold']
print('Final candidate:', FINAL_VARIANT, '| threshold:', threshold)
print('Source OOF confusion matrix at the development-selected threshold')
display(matrix_table(selected['source_classification']))
print('Nested source confusion matrix')
display(matrix_table(selected['nested_source_classification']))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(final_model, OUTPUT_DIR / 'lightgbm_clean13_field_v2.joblib')
metadata = {
    'model_name': 'LightGBM_Clean13_Field_v2', 'variant': 'clean13', 'training_variant': FINAL_VARIANT,
    'features': FEATURES, 'n_features': len(FEATURES), 'positive_label': 1, 'target': 'training_label',
    'positive_meaning': 'Sensor-inferred onset window', 'negative_meaning': 'Selected source background and operator-labelled field negatives',
    'baseline': {'used_by_exported_model': False},
    'classification_threshold': threshold, 'threshold_selection': selected['selection'],
    'threshold_grid': THRESHOLDS.tolist(),
    'classification_policy': 'instantaneous score comparison; voting configured by consumer',
    'augmentation': {'pm_rounding_training_only': FINAL_VARIANT == 'field_augmented13',
                     'raw_source_weight': .5 if FINAL_VARIANT == 'field_augmented13' else 1.,
                     'quantized_source_weight': .5 if FINAL_VARIANT == 'field_augmented13' else 0.,
                     'field_negative_weight': 1., 'inference_measurements_unchanged': True},
    'feature_extraction': {'history_samples': 6, 'rolling_window': 5, 'std_ddof': 1, 'pm_ratio_epsilon': .001},
    'training_field_sessions': sorted(field.loc[train_field, 'source_session'].unique()),
    'threshold_validation_sessions': sorted(field.loc[validation_field, 'source_session'].unique()),
    'heldout_sessions': sorted(field.loc[heldout_field, 'source_session'].unique()),
    'source_sha256': hashlib.sha256(source_path.read_bytes()).hexdigest(),
    'field_sha256': hashlib.sha256(field_path.read_bytes()).hexdigest(),
    'source_unique_rows': int(source_mask.sum()), 'source_unique_positives': int(ys.loc[source_mask].sum()),
    'field_training_rows': int(train_field.sum()), 'parameters': PARAMETERS,
    'physical_fire_ground_truth': 'source onsets are sensor-inferred; field smoke challenge unlabelled',
    'versions': {'python': platform.python_version(), 'lightgbm': lgb.__version__, 'numpy': np.__version__, 'pandas': pd.__version__, 'scikit_learn': sklearn.__version__},
}
report = {
    'experiments': {name: {key: value for key, value in e.items() if key not in ['oof', 'oof_quantized', 'sweep']} for name, e in experiments.items()},
    'field_results': field_result_rows, 'source_group_results_at_reference_threshold': source_table_rows,
    'reference_threshold': REFERENCE_THRESHOLD,
}
(OUTPUT_DIR / 'metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2))
(OUTPUT_DIR / 'results.json').write_text(json.dumps(report, ensure_ascii=False, indent=2))
comparison.to_csv(OUTPUT_DIR / 'comparison.csv', index=False)
field_results.to_csv(OUTPUT_DIR / 'field_results.csv', index=False)
pd.concat(field_score_rows, ignore_index=True).to_csv(OUTPUT_DIR / 'field_scores.csv', index=False)
for name, experiment in experiments.items():
    experiment['sweep'].to_csv(OUTPUT_DIR / f'{name}_threshold_sweep.csv', index=False)
    oof_table = source.loc[experiment['oof'].index, ['source_row_position', 'session_id', 'source_group', 'CNT', 'Fire_Alarm_Transient']].copy()
    oof_table['probability_raw'] = experiment['oof']
    oof_table['probability_quantized_view'] = experiment['oof_quantized']
    oof_table.to_csv(OUTPUT_DIR / f'{name}_source_oof.csv', index=False)
restored = joblib.load(OUTPUT_DIR / 'lightgbm_clean13_field_v2.joblib')
assert list(restored.feature_name_) == FEATURES
np.testing.assert_allclose(restored.predict_proba(Xf.loc[train_field].head(100)), final_model.predict_proba(Xf.loc[train_field].head(100)))
zip_path = OUTPUT_DIR / 'lightgbm_clean13_field_v2_bundle.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for p in sorted(OUTPUT_DIR.iterdir()):
        if p.is_file() and p != zip_path:
            archive.write(p, arcname=p.name)
print('Bundle:', zip_path)
if FileLink is not None:
    display(FileLink(str(zip_path)))


              variant               session                 role  rows  threshold  score_max  score_p99  count_ge_099  direct_positive  vote_3_of_5_positive  false_positive_rate
0      source_clean13         room_20261117       train_negative   900       0.25   0.999522   0.999201           388              392                   312             0.435556
1      source_clean13         cool_20261117       train_negative   636       0.25   0.999541   0.999461           273              295                   259             0.463836
2      source_clean13         warm_20261117       train_negative   275       0.25   0.999500   0.999358           123              141                   138             0.512727
3      source_clean13         room_20261112  validation_negative   630       0.25   0.999196   0.999103           295              295                   248             0.468254
4      source_clean13         room_20261120     heldout_negative    96       0.25   0.999166   0.999166       